# Label Distribution by User

This notebook inspects a padded WritingRing dataset and summarizes the distribution of labels for each user.

Usage:

1. Set `ROOT` to the padded dataset root, its `segmentation_padded/` directory, or a parent directory that resolves to exactly one padded dataset.
2. Set `SELECTED_LABELS = None` to include all labels.
3. Or set `SELECTED_LABELS` to a tuple/list such as `("a", "b", "c")` to inspect only those labels.
4. Run all cells.

The notebook uses the repository's canonical `load_acceleration_data(...)` loader, so it validates the padded packages before computing statistics.


## 1. Imports and repository setup

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd


def find_repository_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "snn").is_dir() and (candidate / "scripts").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not locate repository root containing snn/ and scripts/."
    )


REPOSITORY_ROOT = find_repository_root(Path.cwd())

if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))

from snn.accel_reconstruction_eval import load_acceleration_data

print("Repository root:", REPOSITORY_ROOT)


## 2. Configuration

In [ ]:
# You may point ROOT at:
#   - the exact segmentation_padded/ directory, or
#   - its parent experiment directory, provided it resolves to exactly one padded dataset.
ROOT = Path("outputs/action0_rectified/low-pass/aligned-board-events")

# None -> keep all labels.
# Example:
# SELECTED_LABELS = ("a", "b", "c")
SELECTED_LABELS = None


## 3. Load and validate the padded dataset

In [ ]:
data = load_acceleration_data(
    ROOT,
    repository_root=REPOSITORY_ROOT,
    require_reconstruction=False,
)

manifest = data.sample_manifest.copy()
manifest["user"] = manifest["user"].astype(str)
manifest["label"] = manifest["label"].astype(str)

print("Resolved padded root:", data.padded_root)
print("Users:", manifest["user"].nunique())
print("Labels:", manifest["label"].nunique())
print("Segments:", len(manifest))


## 4. Optional label selection

In [ ]:
if SELECTED_LABELS is None:
    selected_manifest = manifest.copy()
    selected_labels = sorted(selected_manifest["label"].unique())
else:
    selected_labels = [str(label) for label in SELECTED_LABELS]

    if len(selected_labels) != len(set(selected_labels)):
        raise ValueError(f"SELECTED_LABELS contains duplicates: {selected_labels}")

    available_labels = set(manifest["label"])
    missing_labels = [
        label for label in selected_labels
        if label not in available_labels
    ]
    if missing_labels:
        raise ValueError(
            "Selected labels are not present in the dataset: "
            f"{missing_labels}"
        )

    selected_manifest = manifest.loc[
        manifest["label"].isin(selected_labels)
    ].copy()

print("Selected labels:", selected_labels)
print("Selected users:", selected_manifest["user"].nunique())
print("Selected segments:", len(selected_manifest))


## 5. Segment counts: user × label

Rows are users, columns are labels, and each cell is the number of retained padded segments for that user/label pair.


In [ ]:
counts = pd.crosstab(
    selected_manifest["user"],
    selected_manifest["label"],
)

# Preserve SELECTED_LABELS order when the user explicitly provides one.
if SELECTED_LABELS is not None:
    counts = counts.reindex(columns=selected_labels, fill_value=0)

counts["TOTAL"] = counts.sum(axis=1)

display(counts)


## 6. Within-user percentages

Each row sums to 100% across the selected labels. This is useful for checking whether one user's class distribution is strongly imbalanced.


In [ ]:
label_count_columns = [column for column in counts.columns if column != "TOTAL"]

percentages = (
    counts[label_count_columns]
    .div(counts["TOTAL"].replace(0, pd.NA), axis=0)
    .mul(100.0)
)

display(percentages.round(2))


## 7. Label coverage summary

This summarizes how many segments each selected label has in total and how many users contain that label.


In [ ]:
label_summary = (
    selected_manifest.groupby("label")
    .agg(
        total_segments=("sample_id", "size"),
        users_with_label=("user", "nunique"),
    )
)

label_summary["total_users_in_selected_cohort"] = selected_manifest["user"].nunique()
label_summary["user_coverage_percent"] = (
    label_summary["users_with_label"]
    / label_summary["total_users_in_selected_cohort"]
    * 100.0
)

if SELECTED_LABELS is not None:
    label_summary = label_summary.reindex(selected_labels)

display(label_summary)


## 8. Users missing selected labels

For Experiment A, this is particularly useful because the default split validation expects every retained label to appear in train, validation, and test. Labels supported by very few users are therefore risky for a user-disjoint split.


In [ ]:
missing_by_user = counts[label_count_columns].eq(0)

missing_selected_labels = pd.DataFrame({
    "missing_label_count": missing_by_user.sum(axis=1),
    "missing_labels": missing_by_user.apply(
        lambda row: [
            label for label, is_missing in row.items()
            if bool(is_missing)
        ],
        axis=1,
    ),
})

display(missing_selected_labels)


## 9. Visualization

In [ ]:
ax = counts[label_count_columns].plot(
    kind="bar",
    stacked=True,
    figsize=(max(10, len(counts) * 0.7), 6),
)

ax.set_title("Selected label distribution by user")
ax.set_xlabel("User")
ax.set_ylabel("Segment count")
ax.legend(title="Label", bbox_to_anchor=(1.02, 1), loc="upper left")

plt.tight_layout()
plt.show()


## 10. Optional: export tables

Uncomment the lines below if you want CSV outputs next to the notebook or in another directory.


In [ ]:
# OUTPUT_DIR = REPOSITORY_ROOT / "notebooks" / "artifacts" / "label_distribution"
# OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
#
# counts.to_csv(OUTPUT_DIR / "user_label_counts.csv")
# percentages.to_csv(OUTPUT_DIR / "user_label_percentages.csv")
# label_summary.to_csv(OUTPUT_DIR / "label_summary.csv")
# missing_selected_labels.to_csv(
#     OUTPUT_DIR / "users_missing_selected_labels.csv"
# )
